<a href="https://colab.research.google.com/github/mari-muthu-k/agent-to-production/blob/main/notebooks/day1/Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 1 Demo: How LLMs work

Needs `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL`.

In [1]:
!pip install -q tiktoken openai

In [2]:
import os
try:
    from google.colab import userdata
    for name in ["LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL"]:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass
except ImportError:
    pass

import tiktoken
enc = tiktoken.get_encoding("o200k_base")   # GPT-4o-era tokenizer
print("tokenizer loaded")

tokenizer loaded


---
## D3.1 What the model actually receives

In [14]:
text = "Researchers trained a small code model."
ids = enc.encode(text)
print("text:     ", text)
print("token IDs:", ids)
print("pieces:   ", [enc.decode([i]) for i in ids])
print(f"\n{len(text)} characters -> {len(ids)} tokens")

text:      Researchers trained a small code model.
token IDs: [123315, 22203, 261, 3291, 3490, 2359, 13]
pieces:    ['Researchers', ' trained', ' a', ' small', ' code', ' model', '.']

39 characters -> 7 tokens


Note : common words are single tokens; the space attaches to the word **after** it.

---
## D3.2 Same text, three tokenizers  ·

**Before running:** ask the room to predict in chat which row costs the most tokens.

In [ ]:
encoders = {name: tiktoken.get_encoding(name) for name in ["gpt2", "cl100k_base", "o200k_base"]}
samples = {
    "English":   "Researchers trained a small code model.",
    "AI jargon": "LoRA fine-tuning of a Mixture-of-Experts transformer",
    "Python":    "def forward(self, x):\n    return self.attn(x)",
    "Tamil":     "ஆராய்ச்சியாளர்கள் ஒரு சிறிய மாதிரியை பயிற்றுவித்தனர்.",
}
print(f"{'text':10}" + "".join(f"{n:>14}" for n in encoders))
for label, t in samples.items():
    print(f"{label:10}" + "".join(f"{len(e.encode(t)):>14}" for e in encoders.values()))

Takeaways to say out loud:
1. Newer tokenizers use **fewer tokens** for the same text (bigger vocabularies).
2. **Jargon and acronyms** split into pieces.
3. **Non-English text** costs far more for the same meaning: English-based cost estimates are wrong for Tamil or Hindi users.

Optional: show the AI-jargon split piece by piece.

In [ ]:
jargon = samples["AI jargon"]
for name, e in encoders.items():
    print(f"{name:12}", [e.decode([i]) for i in e.encode(jargon)])

---
## D3.3 Code is tokens too

In [ ]:
snippet = """def forward(self, x):
    attn = self.attn(x)
    return self.proj(attn)"""
pieces = [enc.decode([i]) for i in enc.encode(snippet)]
print(len(pieces), "tokens:")
print(pieces)

Point at: `def`, `return` and `self` are single tokens; indentation is its own tokens.

---
## D3.4 Why models miscount letters (needs the API)

In [ ]:
word = "strawberry"
print("the model sees:", [enc.decode([i]) for i in enc.encode(word)])
try:
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.environ["LLM_BASE_URL"])
    r = client.chat.completions.create(model=os.environ["LLM_MODEL"], max_tokens=20, temperature=0,
        messages=[{"role": "user", "content": f"How many letter r's are in '{word}'? Answer with just a number."}])
    print("model says:", r.choices[0].message.content.strip())
except Exception as e:
    print("(API not configured:", type(e).__name__, ")")
print("code says: ", word.count("r"))

Lesson: counting and spelling belong in **code**, not in the model.

---
## D3.5 Does a whole paper fit?  ·  *slide: "Context is a budget"*

In [ ]:
words_in_paper = 8_000                      # a typical AI paper
tokens_per_word = 1 / 0.75                  # rule of thumb for English
paper_tokens = int(words_in_paper * tokens_per_word)
price_in_per_1m = 0.50                      # illustrative; use your gateway's price

print(f"~{paper_tokens:,} tokens for one paper")
for questions in (1, 10, 50):
    cost = paper_tokens * questions * price_in_per_1m / 1e6
    print(f"{questions:>3} questions, resending the whole paper each time: ~{paper_tokens * questions:>9,} input tokens  ≈ ${cost:.3f}")
print("\nThe model has no memory, so a naive app resends the paper every call. Day 2: send only the relevant parts.")